# Notebook 2: Energy Conservation Validation

Compares leapfrog vs. RK4 vs. Forward Euler on a harmonic oscillator.
Produces Fig 3 in the paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hamiltonian_modal.world_model.symplectic import integrate_trajectory

n_modes = 5
omega_sq = np.array([1.0, 4.0, 9.0, 16.0, 25.0])
eta0 = np.array([1.0, 0.5, 0.3, 0.2, 0.1])
eta_dot0 = np.zeros(n_modes)

def grad_fn(eta, m):
    return omega_sq * eta

def H_fn(eta, eta_dot, m):
    return 0.5 * float(eta_dot @ eta_dot) + 0.5 * float(eta @ (omega_sq * eta))

h, n_steps = 0.01, 5000
results = {}
for integrator in ['leapfrog', 'rk4', 'euler']:
    results[integrator] = integrate_trajectory(grad_fn, H_fn, eta0, eta_dot0, 0, h, n_steps, integrator)

H0 = H_fn(eta0, eta_dot0, 0)
print(f'Initial energy: {H0:.4f}')
for name, r in results.items():
    drift = abs(r['energy'][-1] - H0) / abs(H0)
    print(f'{name:10s}  final drift: {drift:.4%}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'leapfrog': 'tab:blue', 'rk4': 'tab:orange', 'euler': 'tab:red'}
labels = {'leapfrog': 'Leapfrog (symplectic)', 'rk4': 'RK4', 'euler': 'Forward Euler'}
for name, r in results.items():
    drift = np.abs(r['energy'] - H0) / abs(H0)
    ax.semilogy(r['time'], np.maximum(drift, 1e-14), color=colors[name], linewidth=2, label=labels[name])
t = np.linspace(0, n_steps * h, 200)
ax.plot(t, 0.001 * h**2 * t, 'k--', linewidth=1.5, label=r'$Ch^2t$ bound')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Relative energy drift')
ax.set_title('Fig 3: Energy Conservation — Leapfrog vs. RK4 vs. Euler (5-mode HO)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
import pathlib; pathlib.Path('../outputs').mkdir(exist_ok=True)
plt.savefig('../outputs/energy_conservation.png', dpi=150, bbox_inches='tight')
plt.show()